# POLSCI 3

## Going Further: Making Graphs with `ggplot2`

**This notebook is optional and not tied to any particular week.** Do it whenever you have time: it teaches you how to make the kinds of graphs you see in papers and the news.

### What you'll learn

| Tool | What it does |
|---|---|
| `ggplot()` and `aes()` | Sets up a plot and connects variables to the axes |
| `geom_point()`, `geom_smooth()` | Scatterplots and lines of best fit |
| `geom_bar()`, `geom_col()` | Bar graphs |
| `color`, `fill`, `alpha` | Show another variable, or fix points that overlap |
| `labs()` | Titles, subtitles, captions (notes), and axis labels |
| `theme_bw()` and other themes | Change the look of a plot |
| `facet_wrap()`, `facet_grid()` | Split one plot into many panels |
| `geom_hline()`, `scale_y_continuous()` | Reference lines and axis formatting |

### How it works

Each section has (1) an explanation, (2) examples on real data, and (3) exercises using WNBA and AC Transit data. After each exercise, run the check cell like in class to see if you're right. Each problem tells you what to produce and what to name it. In lecture you used `qplot()`, which is a shortcut for `ggplot()`. Here you'll learn the full version.

In [ ]:
library(tidyverse)   # loads ggplot2 (this notebook) and dplyr

wnba <- read_csv('ps3_wnba_teamgames.csv')
ac <- read_csv('ps3_actransit_ridership.csv')

### The data

Datasets:

- `wnba`: one row per WNBA team per game (2022 to 2025, regular season and playoffs). Columns: `season`, `game_id`, `game_date`, `game_type`, `team`, `home_away`, `points`, `opp_points`, `win`, `reg_win_pct`, `assists`, `rebounds`, `turnovers`, `threes_made`.
- `ac`: AC Transit bus ridership, one row per route, per month, per day type (2023 to 2025). Columns: `route`, `type`, `year`, `month`, `day_type`, `avg_daily_riders`, `all_three_days`.

For the examples I made two small summary tables using `dplyr`. If you did the dplyr notebook you'll recognize the code. If not, don't worry about how it works, just run the cell below and look at the results. **Run it before the exercises**, because they use these tables.

- `team.seasons`: one row per team per season, using regular-season games only. `win_pct` is the share of games won, `avg_points` and `avg_threes` are averages per game, and `made_playoffs` is whether the team played in the playoffs that season.
- `ac.monthly`: AC Transit's **system-wide** average daily riders (`riders`) for each month and day type. (Adding up every route's average daily riders gives the total for the whole system.)

In [ ]:
team.seasons <- wnba %>%
  group_by(season, team) %>%
  summarize(win_pct = mean(win[game_type == "regular"]),
            avg_points = mean(points[game_type == "regular"]),
            avg_threes = mean(threes_made[game_type == "regular"]),
            made_playoffs = ifelse(any(game_type == "playoff"), "Yes", "No"),
            .groups = "drop")

team.2025 <- team.seasons %>% filter(season == 2025)

ac.monthly <- ac %>%
  group_by(year, month, day_type) %>%
  summarize(riders = sum(avg_daily_riders), .groups = "drop") %>%
  mutate(month = factor(month, levels = month.abb))   # keeps months in calendar order

team.2025
head(ac.monthly)

---
## 1. Your first plot: a scatterplot

Every `ggplot2` plot is built in layers that you add together with `+`:

1. `ggplot(data, aes(x = ..., y = ...))` sets up the plot. `aes()` (short for *aesthetics*) says which variables go on which axis.
2. A `geom_` function adds the shapes. `geom_point()` draws one point per row, which makes a **scatterplot**.

Put the `+` at the **end** of a line so R knows the plot continues on the next one.

In [ ]:
# Each point is a team in a season: do teams that make more threes win more?
ggplot(team.seasons, aes(x = avg_threes, y = win_pct)) +
  geom_point()

### Your turn

**Q1.** Make a scatterplot of every team-game in `wnba`, with `assists` on the x-axis and `points` on the y-axis. Save it as `q1.plot`. (The last line of the cell, `q1.plot`, shows it.)

In [ ]:
# ggplot() sets up the axes, geom_point() adds one point per team-game
q1.plot <- ggplot(wnba, aes(x = assists, y = points)) +
  geom_point()

q1.plot

In [ ]:
. = ottr::check("tests/q1.R")

---
## 2. More than two variables: color, size, and alpha

You can use color, size, shape, and see-through points to show more information. The key rule:

- **Inside `aes()`:** the look depends on a **variable** in the data (one color per category, for example).
- **Outside `aes()`:** the look is the **same for every point**.

In [ ]:
# Outside aes(): every point is navy and big
ggplot(team.seasons, aes(x = avg_threes, y = win_pct)) +
  geom_point(color = "navy", size = 3)

# Inside aes(): the color depends on whether the team made the playoffs
ggplot(team.seasons, aes(x = avg_threes, y = win_pct, color = made_playoffs)) +
  geom_point(size = 3)

When thousands of points overlap, make them see-through with `alpha` (0 is invisible, 1 is solid) so you can see where they pile up:

In [ ]:
ggplot(wnba, aes(x = threes_made, y = points)) +
  geom_point(alpha = 0.2)

### Your turn

**Q2.** Remake your Q1 scatterplot (`assists` vs. `points`) with the points **colored by `home_away`** and made partly see-through with **`alpha = 0.4`**. Save it as `q2.plot`.

In [ ]:
# color goes INSIDE aes() because it depends on a variable; alpha goes OUTSIDE because it is the same for every point
q2.plot <- ggplot(wnba, aes(x = assists, y = points, color = home_away)) +
  geom_point(alpha = 0.4)

q2.plot

In [ ]:
. = ottr::check("tests/q2.R")

**Q3.** A classmate writes `geom_point(aes(color = "blue"))` to make every point blue. Instead the points come out in R's default pinkish-red color, with a legend that says "blue". What went wrong?

- **A.** R doesn't have a color called blue.
- **B.** The code needs `alpha` to show the color.
- **C.** Inside `aes()`, "blue" is treated like a category in the data, not a color. To make every point blue, put `color = "blue"` outside `aes()`.
- **D.** `color` only works with `geom_bar()`.

Save your choice as a letter (text in quotes) in `q3.answer`.

In [ ]:
q3.answer <- "C"

**Answer: C.** Anything inside aes() is mapped to a variable (here, a single category named "blue"), so ggplot picks its own color for it. To set a fixed look, use color = "blue" outside aes().

In [ ]:
. = ottr::check("tests/q3.R")

---
## 3. A line of best fit

In lecture you added `geom_smooth(method = 'lm')` to a `qplot()` to draw the **line of best fit** (the regression line). It works the same way here. `se = FALSE` removes the gray band around the line.

In [ ]:
ggplot(team.seasons, aes(x = avg_threes, y = win_pct)) +
  geom_point() +
  geom_smooth(method = "lm")

ggplot(team.seasons, aes(x = avg_threes, y = win_pct)) +
  geom_point() +
  geom_smooth(method = "lm", se = FALSE)

### Your turn

**Q4.** Make a scatterplot of every team-game in `wnba` with `threes_made` on the x-axis and `points` on the y-axis, with a **line of best fit** added. Save it as `q4.plot`.

In [ ]:
# points first, then the line of best fit as a second layer
q4.plot <- ggplot(wnba, aes(x = threes_made, y = points)) +
  geom_point(alpha = 0.3) +
  geom_smooth(method = "lm")

q4.plot

In [ ]:
. = ottr::check("tests/q4.R")

---
## 4. Bar graphs

There are two bar-graph geoms, and the difference is **who decides how tall the bars are**:

- `geom_bar()` **counts** the rows in each category for you. You only give it an `x`.
- `geom_col()` draws bars as tall as a **column you provide**. You give it an `x` and a `y`.

In [ ]:
# geom_bar(): R counts how many team-seasons fall in each category
ggplot(team.seasons, aes(x = made_playoffs)) +
  geom_bar()

# geom_col(): I give the heights (average points per game)
ggplot(team.2025, aes(x = team, y = avg_points)) +
  geom_col()

Team names get squished on the bottom axis. Two fixes: `reorder(team, avg_points)` sorts the bars by their height, and `coord_flip()` turns the bars sideways so the names are easy to read.

In [ ]:
ggplot(team.2025, aes(x = reorder(team, avg_points), y = avg_points)) +
  geom_col() +
  coord_flip()

(The axis titles look odd, like `reorder(team, avg_points)`. We'll fix those with `labs()` in Section 5.)

To show a second variable, map it to `fill` (bars are filled; points are colored). With two groups of bars, the default is to **stack** them. `position = "dodge"` puts them side by side. (You can also pipe a filtered dataset straight into `ggplot()`.)

In [ ]:
# stacked (the default)
ac.monthly %>%
  filter(year == 2024) %>%
  ggplot(aes(x = month, y = riders, fill = day_type)) +
  geom_col()

# side by side
ac.monthly %>%
  filter(year == 2024) %>%
  ggplot(aes(x = month, y = riders, fill = day_type)) +
  geom_col(position = "dodge")

### Your turn

**Q5.** Make a bar graph showing the **number of team-games** in each `game_type` in `wnba` (let R do the counting). Save it as `q5.plot`.

In [ ]:
# we want counts, so only give x and let geom_bar() count
q5.plot <- ggplot(wnba, aes(x = game_type)) +
  geom_bar()

q5.plot

In [ ]:
. = ottr::check("tests/q5.R")

**Q6.** Using `team.2025`, make a bar graph of **average points per game** (`avg_points`) for each team, as **horizontal bars sorted so the highest-scoring team is at the top**. Save it as `q6.plot`.

In [ ]:
# we provide the heights, so geom_col(); reorder() sorts the bars; coord_flip() makes them horizontal
q6.plot <- ggplot(team.2025, aes(x = reorder(team, avg_points), y = avg_points)) +
  geom_col() +
  coord_flip()

q6.plot

In [ ]:
. = ottr::check("tests/q6.R")

**Q7.** Make a bar graph of AC Transit's system-wide average daily `riders` for each `month` in **2024**, with the day types (`day_type`) shown as **side-by-side** bars in different colors. Use `ac.monthly`. Save it as `q7.plot`.

In [ ]:
# filter to 2024, map day_type to fill, and use position = "dodge" for side-by-side bars
q7.plot <- ac.monthly %>%
  filter(year == 2024) %>%
  ggplot(aes(x = month, y = riders, fill = day_type)) +
  geom_col(position = "dodge")

q7.plot

In [ ]:
. = ottr::check("tests/q7.R")

---
## 5. Titles, subtitles, captions, and axis labels

A plot should be understandable on its own. `labs()` (short for *labels*) adds all the words:

- `title`: what the plot is about
- `subtitle`: a line with more detail
- `caption`: a small note at the bottom, usually the **data source**
- `x` and `y`: axis labels (use plain English, not variable names!)
- `fill` or `color`: the legend title

Labels are named after the *aesthetic*, so when you use `coord_flip()`, `x` is still the variable you mapped to `x` (it just ends up on the vertical axis).

In [ ]:
ggplot(team.2025, aes(x = reorder(team, avg_points), y = avg_points)) +
  geom_col() +
  coord_flip() +
  labs(title = "Scoring in the WNBA, 2025 regular season",
       subtitle = "Average points per game by team",
       caption = "Source: ESPN box scores, compiled by the wehoop project",
       x = "Team",
       y = "Average points per game")

### Your turn

**Q8.** Remake (or build on) your Q7 plot and add:

- a **title**, a **subtitle**, and a **caption** (a note about the data source)
- plain-English labels for the **x-axis**, the **y-axis**, and the **legend** (the legend title is `fill`)

Save it as `q8.plot`. Your labels can say whatever you like, but they shouldn't just be the variable names.

In [ ]:
q8.plot <- ac.monthly %>%
  filter(year == 2024) %>%
  ggplot(aes(x = month, y = riders, fill = day_type)) +
  geom_col(position = "dodge") +
  labs(title = "AC Transit ridership by month, 2024",
       subtitle = "Weekdays carry far more riders than weekends all year",
       caption = "Source: AC Transit ridership data",
       x = "Month",
       y = "Average daily riders (whole system)",
       fill = "Day type")

q8.plot

In [ ]:
. = ottr::check("tests/q8.R")

---
## 6. Themes

The gray background is ggplot's default look. A **theme** changes the look of the whole plot: background, grid lines, borders, fonts. Add one with `+`:

| Theme | Look |
|---|---|
| `theme_gray()` | the default: gray background, white grid lines |
| `theme_bw()` | white background, light gray grid lines, thin border |
| `theme_minimal()` | no background or border, light grid lines |
| `theme_classic()` | no grid lines, just the x and y axis lines |
| `theme_light()` | light gray lines and border |
| `theme_dark()` | dark background |

You can also fine-tune with `theme()`, for example `theme(legend.position = "bottom")` (other options: `"top"`, `"left"`, `"none"`) or `theme(plot.title = element_text(face = "bold"))`. If axis labels are crowded, `theme(axis.text.x = element_text(angle = 90, hjust = 1))` rotates them.

**Order matters:** add the full theme (like `theme_bw()`) **first** and your `theme()` tweaks **after**, or the full theme will undo your tweaks.

In [ ]:
p.base <- ggplot(team.seasons, aes(x = avg_threes, y = win_pct, color = made_playoffs)) +
  geom_point(size = 3) +
  labs(x = "Threes made per game", y = "Win percentage", color = "Made playoffs?")

p.base + theme_bw()

p.base + theme_minimal()

p.base + theme_classic()

p.base + theme_bw() + theme(legend.position = "bottom")

**Q9.** Which theme gives a **white background, light gray grid lines, and a thin border** around the plot?

- **A.** `theme_minimal()`
- **B.** `theme_bw()`
- **C.** `theme_classic()`
- **D.** `theme_gray()`

Save your choice as a letter (text in quotes) in `q9.answer`.

In [ ]:
q9.answer <- "B"

**Answer: B.** theme_bw() is the white-background version of the default, with a border. theme_minimal() has no border, theme_classic() has no grid lines, and theme_gray() is the default gray look.

In [ ]:
. = ottr::check("tests/q9.R")

**Q10.** Take your Q2 plot (points colored by `home_away`), switch it to **`theme_bw()`**, and move the legend to the **bottom**. Save it as `q10.plot`.

In [ ]:
# the full theme goes first, then the tweak
q10.plot <- ggplot(wnba, aes(x = assists, y = points, color = home_away)) +
  geom_point(alpha = 0.4) +
  theme_bw() +
  theme(legend.position = "bottom")

q10.plot

In [ ]:
. = ottr::check("tests/q10.R")

---
## 7. Facets: one plot, many panels

Sometimes one plot gets crowded. **Facets** split it into small panels, one for each category, using the same axes so you can compare.

- `facet_wrap(~ variable)` makes one panel per category of **one variable** and wraps them into rows to fit the page. `ncol = 2` sets the number of columns.
- `facet_grid(rows ~ columns)` puts one variable in the **rows** and another in the **columns**, so every row and column means something. Use a `.` if you only want one of them (`facet_grid(. ~ season)`).

Panels share the same y-axis scale by default. If the groups are on very different scales, add `scales = "free_y"` (or `"free_x"`, or `"free"`) to let each panel have its own.

In [ ]:
p.base <- ggplot(team.seasons, aes(x = avg_threes, y = win_pct)) +
  geom_point() +
  theme_bw()

# one panel per season
p.base + facet_wrap(~ season)

# the same, but 2 columns
p.base + facet_wrap(~ season, ncol = 2)

# rows = made the playoffs or not, columns = season
p.base + facet_grid(made_playoffs ~ season)

### Your turn

**Q11.** Make your Q1 scatterplot (`assists` vs. `points` for every team-game) with one panel per `season`, arranged in **2 columns** using `facet_wrap()`. Save it as `q11.plot`.

In [ ]:
q11.plot <- ggplot(wnba, aes(x = assists, y = points)) +
  geom_point(alpha = 0.3) +
  facet_wrap(~ season, ncol = 2)

q11.plot

In [ ]:
. = ottr::check("tests/q11.R")

**Q12.** Using `ac.monthly`, make a bar graph of system-wide average daily `riders` by `month`, with **one row of panels for each `day_type` and one column for each `year`** (use `facet_grid()`). Save it as `q12.plot`.

In [ ]:
# rows ~ columns: day_type down the side, year across the top
q12.plot <- ggplot(ac.monthly, aes(x = month, y = riders)) +
  geom_col() +
  facet_grid(day_type ~ year)

q12.plot

In [ ]:
. = ottr::check("tests/q12.R")

**Q13.** Make a bar graph of the **number of team-games in each `season`** (let R count them), with a separate panel for each `game_type`. The regular season has about 10 times more games than the playoffs, so let each panel have **its own y-axis scale**. Save it as `q13.plot`.

In [ ]:
# geom_bar() counts; facet_wrap() splits by game type; scales = "free_y" lets each panel pick its own y-axis
q13.plot <- ggplot(wnba, aes(x = season)) +
  geom_bar() +
  facet_wrap(~ game_type, scales = "free_y")

q13.plot

In [ ]:
. = ottr::check("tests/q13.R")

**Q14.** What is the main difference between `facet_wrap()` and `facet_grid()`?

- **A.** `facet_grid()` can only split by one variable.
- **B.** `facet_wrap()` always puts every panel in a single row.
- **C.** They are identical and just have different names.
- **D.** `facet_wrap()` makes a ribbon of panels, one per category, and wraps it to fit the page. `facet_grid()` puts one variable in the rows and another in the columns, so the layout itself has meaning.

Save your choice as a letter (text in quotes) in `q14.answer`.

In [ ]:
q14.answer <- "D"

**Answer: D.** Use facet_wrap() when you want one panel per category of a variable. Use facet_grid() when you want to compare across two variables at once, with one variable down the rows and the other across the columns.

In [ ]:
. = ottr::check("tests/q14.R")

---
## 8. Reference lines and axis formatting

**Reference lines** help readers judge a pattern: `geom_hline(yintercept = ...)` draws a horizontal line, `geom_vline(xintercept = ...)` a vertical one, and `geom_abline(slope = 1, intercept = 0)` a 45-degree line. Use `linetype` and `color` to style them.

**Axis formatting** goes in `scale_` functions. For example, `scale_y_continuous(labels = scales::percent)` shows `0.5` as `50%`, and `scale_x_continuous(labels = scales::comma)` puts commas in big numbers.

To save a finished plot as an image, use `ggsave("my-plot.png", plot.name, width = 7, height = 4)`.

In [ ]:
ggplot(team.seasons, aes(x = avg_threes, y = win_pct)) +
  geom_point() +
  geom_hline(yintercept = 0.5, linetype = "dashed", color = "red") +
  scale_y_continuous(labels = scales::percent)

### Your turn

**Q15.** Make a scatterplot of `team.seasons` with `avg_threes` on the x-axis and `win_pct` on the y-axis, add a **dashed red horizontal line at 0.5** (a .500 record), and show the y-axis as **percentages**. Save it as `q15.plot`.

In [ ]:
q15.plot <- ggplot(team.seasons, aes(x = avg_threes, y = win_pct)) +
  geom_point() +
  geom_hline(yintercept = 0.5, linetype = "dashed", color = "red") +
  scale_y_continuous(labels = scales::percent)

q15.plot

In [ ]:
. = ottr::check("tests/q15.R")

### Challenge

**Q16.** Put it all together. Make a bar graph of AC Transit's system-wide average daily `riders` by `month` for **2024**, with **one panel for each `day_type`** (`facet_wrap()`), using **`theme_minimal()`**. Add a title, subtitle, caption, and plain-English axis labels. The panels already say which day type is which, so **remove the legend** if you added one. Save it as `q16.plot`.

In [ ]:
q16.plot <- ac.monthly %>%
  filter(year == 2024) %>%
  ggplot(aes(x = month, y = riders, fill = day_type)) +
  geom_col() +
  facet_wrap(~ day_type) +
  labs(title = "AC Transit ridership by month and day type, 2024",
       subtitle = "Weekdays carry far more riders, and weekday ridership dips in the summer",
       caption = "Source: AC Transit ridership data",
       x = "Month",
       y = "Average daily riders (whole system)") +
  theme_minimal() +
  theme(legend.position = "none")

q16.plot

In [ ]:
. = ottr::check("tests/q16.R")

---
## What you learned

- `ggplot(data, aes(x, y))` + a `geom_`: build plots in layers
- `geom_point()`, `geom_smooth(method = "lm")`: scatterplots and lines of best fit
- `geom_bar()` (counts) vs. `geom_col()` (heights you provide), plus `reorder()`, `coord_flip()`, and `position = "dodge"`
- `color`, `fill`, `alpha`: inside `aes()` for variables, outside for fixed looks
- `labs()`: title, subtitle, caption, axis labels, legend titles
- Themes: `theme_bw()`, `theme_minimal()`, `theme_classic()`, and `theme()` tweaks
- `facet_wrap()` and `facet_grid()`: split a plot into panels (and `scales = "free_y"`)
- `geom_hline()` and `scale_y_continuous()`: reference lines and axis formatting

A good graph is one a reader can understand without you standing next to it.